In [0]:
USE ecommerce_lakehouse;

-- 1. Clean & Deduplicate Customers
CREATE OR REPLACE TABLE silver_customers AS
WITH ranked_customers AS (
    SELECT 
        TRIM(customer_id) AS customer_id,
        CAST(signup_date AS DATE) AS signup_date,
        TRIM(country) AS country,
        ROW_NUMBER() OVER (
            PARTITION BY TRIM(customer_id) 
            ORDER BY signup_date ASC
        ) AS row_num
    FROM bronze_customers
    WHERE customer_id IS NOT NULL
)
SELECT 
    customer_id,
    signup_date,
    country
FROM ranked_customers
WHERE row_num = 1;


-- 2. Clean & Deduplicate Products
CREATE OR REPLACE TABLE silver_products AS
WITH ranked_products AS (
    SELECT 
        TRIM(product_id) AS product_id,
        TRIM(category) AS category,
        CAST(price AS DECIMAL(10,2)) AS price,
        ROW_NUMBER() OVER (
            PARTITION BY TRIM(product_id) 
            ORDER BY price DESC
        ) AS row_num
    FROM bronze_products
    WHERE product_id IS NOT NULL
)
SELECT 
    product_id,
    category,
    price
FROM ranked_products
WHERE row_num = 1;


-- 3. Clean & Deduplicate Orders
CREATE OR REPLACE TABLE silver_orders AS
WITH ranked_orders AS (
    SELECT 
        TRIM(order_id) AS order_id,
        TRIM(customer_id) AS customer_id,
        TRIM(product_id) AS product_id,
        CAST(order_date AS DATE) AS order_date,
        CAST(total_amount AS DECIMAL(10,2)) AS total_amount,
        TRIM(status) AS status,
        ROW_NUMBER() OVER (
            PARTITION BY TRIM(order_id) 
            ORDER BY order_date DESC
        ) AS row_num
    FROM bronze_orders
    WHERE order_id IS NOT NULL 
      AND status = 'Completed' -- Filters out cancelled or invalid orders
)
SELECT 
    order_id,
    customer_id,
    product_id,
    order_date,
    total_amount,
    status
FROM ranked_orders
WHERE row_num = 1;